<a href="https://colab.research.google.com/github/alicedambroz/PRISM-dataset/blob/main/SWAT_Weather.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**SWAT+ model for High Island Creek Watershed**

```
Created on: '2026-02-17'
Updated on: '2026-03-17'
Alice Dambroz | apratesb@umn.edu
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#PRISM dataset
```
Precipitation (mm)
Maximum and minimum temperatures (°C)
Relative humidity (%)
```

#For weather points

From PRISM dataset. Stations are based on 69 points over the watershed (location provided by Brent).

In [ ]:
import pandas as pd
import numpy as np
import os
from google.colab import drive

# 1. MOUNT GOOGLE DRIVE
drive.mount('/content/drive')

# --- CONFIGURATION ---
input_file = '/content/drive/My Drive/SWAT_High-Island/SWAT_PRISM_69_Stations_Master.csv'
output_base = '/content/drive/My Drive/SWAT_High-Island/SWAT_Inputs'

# Create subfolders for organization
folders = ['pcp', 'tmp', 'rhd']
for f in folders:
    os.makedirs(os.path.join(output_base, f), exist_ok=True)

# --- 2. LOAD MASTER DATA ---
print("Loading master file... this may take a moment.")
df = pd.read_csv(input_file)

# --- 3. CALCULATE RELATIVE HUMIDITY ---
print("Calculating Average Temperature and Relative Humidity...")

# Calculate T_mean
df['tmean'] = (df['tmax'] + df['tmin']) / 2

# Apply the RH formula
def calculate_rh(t, td):
    numerator = np.exp((17.625 * td) / (243.04 + td))
    denominator = np.exp((17.625 * t) / (243.04 + t))
    rh = 100 * (numerator / denominator)
    return np.clip(rh, 0, 100) # Cap between 0-100%

df['relh'] = calculate_rh(df['tmean'], df['tdmean'])

# --- 4. DATA FACTORY: GENERATE SWAT+ FILES ---
# Get unique stations based on your ID column (check if it's 'station_id' or 'ID')
station_col = 'id'
stations = df[station_col].unique()

print(f"Processing {len(stations)} stations...")

for st_id in stations:
    # Filter data for this specific station and sort by date
    st_data = df[df[station_col] == st_id].sort_values('date_str')

    # Get the start date for the header (e.g., 19810101)
    start_date_str = str(int(st_data['date_str'].iloc[0]))

    # Clean station name for filename (remove spaces/special chars)
    safe_name = str(st_id).replace(" ", "_")

    # A. PRECIPITATION (.pcp)
    with open(os.path.join(output_base, 'pcp', f'st_{safe_name}.pcp'), 'w') as f:
        f.write(f"{start_date_str}\n")
        np.savetxt(f, st_data['ppt'].values, fmt='%.3f')

    # B. TEMPERATURE (.tmp)
    # SWAT+ temperature files usually have Tmax, Tmin on the same line
    with open(os.path.join(output_base, 'tmp', f'st_{safe_name}.tmp'), 'w') as f:
        f.write(f"{start_date_str}\n")
        # Stack tmax and tmin side-by-side
        tmp_data = st_data[['tmax', 'tmin']].values
        np.savetxt(f, tmp_data, fmt='%.3f,%.3f')

    # C. RELATIVE HUMIDITY (.rhd)
    with open(os.path.join(output_base, 'rhd', f'st_{safe_name}.rhd'), 'w') as f:
        f.write(f"{start_date_str}\n")
        # RH values are decimal (0.0 to 1.0) in some SWAT versions,
        # but 0-100 in others. Most SWAT+ editors expect 0-1 (fraction).
        # Let's use fraction (0.85 instead of 85%). If your editor needs 0-100, remove the /100.
        np.savetxt(f, st_data['relh'].values / 100, fmt='%.4f')

print(f"Done! Files are located in: {output_base}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading master file... this may take a moment.
Calculating Average Temperature and Relative Humidity...
Processing 69 stations...
Done! Files are located in: /content/drive/My Drive/SWAT_High-Island/SWAT_Inputs


In [ ]:
import pandas as pd
import os

# --- 1. CONFIGURATION ---
# Point this to your PRISM Master File (which contains the coordinates)
input_file = '/content/drive/My Drive/SWAT_High-Island/SWAT_PRISM_69_Stations_Master.csv'
output_folder = '/content/drive/My Drive/SWAT_High-Island/SWAT_weather_files'
output_file = os.path.join(output_folder, 'station_locations.txt')

print("Loading Master Data to extract coordinates...")
df = pd.read_csv(input_file)

Loading Master Data to extract coordinates...


In [ ]:
df.head(100)

,id,date_str,year,doy,ppt,tmax,tmin,tdmean,longitude,latitude
0,1,19810101,1981,1,0.0,-1.251,-14.693000,-11.117000,-94.677270,44.695157
1,2,19810101,1981,1,0.0,-1.501,-14.655000,-11.475000,-94.450717,44.694979
2,3,19810101,1981,1,0.0,-1.367,-14.544000,-11.573000,-94.508515,44.716744
3,4,19810101,1981,1,0.0,-1.367,-14.544000,-11.573000,-94.536835,44.705551
4,5,19810101,1981,1,0.0,-1.339,-14.593000,-11.466000,-94.608934,44.701627
...,...,...,...,...,...,...,...,...,...,...
95,27,19810102,1981,2,0.0,-6.830,-23.198999,-17.611000,-94.058869,44.657291
96,28,19810102,1981,2,0.0,-6.645,-22.857000,-17.450001,-94.143904,44.647218
97,29,19810102,1981,2,0.0,-5.885,-22.447001,-17.236000,-94.353209,44.666913
98,30,19810102,1981,2,0.0,-6.068,-22.281000,-16.648001,-94.614316,44.662575


In [ ]:
import pandas as pd
import os

# --- 1. CONFIGURATION ---
# Point this to your PRISM Master File (which contains the coordinates)
input_file = '/content/drive/My Drive/SWAT_High-Island/SWAT_PRISM_69_Stations_Master.csv'
output_folder = '/content/drive/My Drive/SWAT_High-Island/SWAT_weather_files'
output_file = os.path.join(output_folder, 'station_locations.txt')

print("Loading Master Data to extract coordinates...")
df = pd.read_csv(input_file)

# --- 2. IDENTIFY COLUMNS ---
# This safely finds your ID, Latitude, and Longitude columns regardless of capitalization
id_col = [col for col in df.columns if col.lower() in ['id', 'station_id']][0]
lat_col = [col for col in df.columns if 'lat' in col.lower()][0]
lon_col = [col for col in df.columns if 'lon' in col.lower()][0]

print(f"Using columns -> ID: '{id_col}', LAT: '{lat_col}', LON: '{lon_col}'")

# --- 3. EXTRACT UNIQUE STATIONS ---
# Drop all the duplicate daily rows so we just have one row per station
stations_df = df.drop_duplicates(subset=[id_col]).copy()

# --- 4. FORMAT FOR SWAT+ ---
# Create the 'NAME' column to match the file names we generated earlier (e.g., st_1)
stations_df['NAME'] = 'st_' + stations_df[id_col].astype(str)

# Add the default Elevation
stations_df['ELEVATION'] = 300

# Rename columns to perfectly match SWAT+ requirements
stations_df = stations_df.rename(columns={
    id_col: 'ID',
    lat_col: 'LAT',
    lon_col: 'LONG'
})

# Reorder the columns to standard SWAT+ format
final_stations_df = stations_df[['ID', 'NAME', 'LAT', 'LONG', 'ELEVATION']]

# Sort numerically by ID just to keep it neat
final_stations_df = final_stations_df.sort_values(by='ID')

# --- 5. EXPORT TO TXT ---
# SWAT+ expects a comma-separated file, so we export as a CSV but name it .txt
final_stations_df.to_csv(output_file, index=False)

print(f"\nSUCCESS! Station location file saved to: {output_file}")
print("\nPreview of your first 5 stations:")
print(final_stations_df.head().to_string(index=False))

#Average weather files

For PRISM dataset based average values for the watershed polygon.

In [ ]:
import pandas as pd
import os

# --- CONFIGURATION ---
input_csv = '/content/drive/My Drive/SWAT_High-Island/SWAT_PRISM_Weather_Data.csv'
output_folder = '/content/drive/My Drive/SWAT_High-Island/SWAT_weather_files'
station_name = 'watershed_avg'

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Load data
print(f"File: {input_csv}")
df = pd.read_csv(input_csv)

# Check if sorted by date
df = df.sort_values(by=['date_str'])

# Get Start Date (same format as before)
start_date = str(df['date_str'].iloc[0])

print(f"Station {station_name}, first records {start_date}...")

# --- CREATE .PCP FILE ---
pcp_filename = os.path.join(output_folder, f"{station_name}.pcp")
with open(pcp_filename, 'w') as f:
    f.write(f"{start_date}\n")
    for val in df['ppt']:
        f.write(f"{val:.2f}\n")

# --- CREATE .TMP FILE ---
tmp_filename = os.path.join(output_folder, f"{station_name}.tmp")
with open(tmp_filename, 'w') as f:
    f.write(f"{start_date}\n")
    for _, row in df.iterrows():
        f.write(f"{row['tmax']:.2f},{row['tmin']:.2f}\n")

print("Done")

#Plots to check weather data

To check if there is anything strange with the dataset.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

# --- 1. CONFIGURATION ---
input_file = '/content/drive/My Drive/SWAT_High-Island/SWAT_PRISM_69_Stations_Master.csv'
output_folder = '/content/drive/My Drive/SWAT_High-Island/SWAT_weather_files'
os.makedirs(output_folder, exist_ok=True)
sns.set_theme(style="whitegrid")

print("Loading 69-station master file...")
df = pd.read_csv(input_file)

# --- 2. PREPARE DATA ---
print("Formatting dates and calculating Relative Humidity...")
df['date'] = pd.to_datetime(df['date_str'].astype(str), format='%Y%m%d')

# Calculate Tmean and Relative Humidity so we can plot it!
df['tmean'] = (df['tmax'] + df['tmin']) / 2
numerator = np.exp((17.625 * df['tdmean']) / (243.04 + df['tdmean']))
denominator = np.exp((17.625 * df['tmean']) / (243.04 + df['tmean']))
df['relh'] = np.clip(100 * (numerator / denominator), 0, 100)

# Make sure station_id is treated as a category, not a continuous number
df['id'] = df['id'].astype(str)

# --- 3. BOX PLOTS (4x1 Grid for 69 Stations) ---
print("Generating Boxplots...")
fig, axes = plt.subplots(4, 1, figsize=(16, 16))
fig.suptitle('Distribution Across 69 Stations (Daily Data)', fontsize=18, y=0.98)

# Define plotting helper to keep code clean
def plot_ensemble_box(ax, var, title, color, log_scale=False):
    sns.boxplot(x='id', y=var, data=df, ax=ax, color=color, fliersize=1)
    ax.set_title(title)
    ax.set_xlabel('69 Individual Stations (Labels Hidden)')
    ax.set_xticks([]) # Hide x-labels because 69 will overlap and look terrible
    if log_scale:
        ax.set_yscale('log')
        # Add a tiny amount to avoid log(0) errors for precipitation
        df[f'{var}_log'] = df[var] + 0.1
        sns.boxplot(x='id', y=f'{var}_log', data=df, ax=ax, color=color, fliersize=1)

plot_ensemble_box(axes[0], 'ppt', 'Precipitation (mm) - Linear', 'skyblue')
plot_ensemble_box(axes[1], 'tmax', 'Max Temperature (°C)', 'tomato')
plot_ensemble_box(axes[2], 'tmin', 'Min Temperature (°C)', 'cornflowerblue')
plot_ensemble_box(axes[3], 'relh', 'Relative Humidity (%)', 'mediumpurple')

plt.tight_layout(pad=2.0)
save_path_box = os.path.join(output_folder, '69_Stations_Boxplots.png')
plt.savefig(save_path_box, dpi=300)
print(f"Saved: {save_path_box}")
plt.show()

# --- 4. TIME SERIES (Monthly Averages) ---
print("Resampling data for monthly time series...")
# Resample by month for EACH station
df_monthly = df.set_index('date').groupby('id').resample('M').mean(numeric_only=True).reset_index()

# Calculate the Watershed Average (Mean of all 69 stations per month)
df_avg = df_monthly.groupby('date').mean(numeric_only=True).reset_index()

fig2, axes2 = plt.subplots(4, 1, figsize=(16, 16), sharex=True)
fig2.suptitle('Temporal Trends: 69 Stations vs. Watershed Average (Monthly)', fontsize=18, y=0.98)

def plot_ensemble_ts(ax, var, ylabel, color_spaghetti, color_mean):
    # Plot all 69 stations as faint lines (spaghetti plot)
    sns.lineplot(data=df_monthly, x='date', y=var, hue='id',
                 ax=ax, palette=[color_spaghetti]*69, alpha=0.15, legend=False)
    # Overlay the Watershed Average as a bold line
    sns.lineplot(data=df_avg, x='date', y=var,
                 ax=ax, color=color_mean, linewidth=2.5, label='Watershed Average')
    ax.set_ylabel(ylabel)

plot_ensemble_ts(axes2[0], 'ppt', 'Avg Precip (mm/day)', 'blue', 'navy')
plot_ensemble_ts(axes2[1], 'tmax', 'Max Temp (°C)', 'red', 'darkred')
plot_ensemble_ts(axes2[2], 'tmin', 'Min Temp (°C)', 'lightblue', 'blue')
plot_ensemble_ts(axes2[3], 'relh', 'Relative Humidity (%)', 'purple', 'indigo')

plt.tight_layout(pad=2.0)
save_path_ts = os.path.join(output_folder, '69_Stations_TimeSeries.png')
plt.savefig(save_path_ts, dpi=300)
print(f"Saved: {save_path_ts}")
plt.show()

print("All visual quality checks complete!")

#IEM ASOS dataset
```
Wind speed (mph -> m/s)
Relative humidity (%)
```

#IEM 1
This is to create the plots and visually analyze the data. Besides, it will provide a text file with missing data information.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# --- CONFIGURATION ---
input_file = '/content/drive/My Drive/SWAT_High-Island/ASOS_data.txt'
output_folder = '/content/drive/My Drive/SWAT_High-Island/SWAT_weather_files'

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# --- 1. LOAD AND PRE-PROCESS ---
print("Loading raw data...")
df = pd.read_csv(input_file, parse_dates=['valid'], na_values=['M', 'm', 'Missing'])

# Replace Trace 'T' with 0.001
df.replace({'T': 0.001}, inplace=True)

# Force numeric conversion
df['relh'] = pd.to_numeric(df['relh'], errors='coerce')
df['sped'] = pd.to_numeric(df['sped'], errors='coerce')

# Convert Wind: mph -> m/s
df['wind_ms'] = df['sped'] * 0.44704

# Extract Date
df['date'] = df['valid'].dt.date
df['date'] = pd.to_datetime(df['date'])

# --- 2. APPLY STRICT CLEANING RULES ---
print("Applying cleaning rules...")

# Rule A: RH > 120% -> Treat as Missing (NaN)
extreme_rh_count = len(df[df['relh'] > 120])
print(f"  - Found {extreme_rh_count} records with RH > 120%. Setting to NaN.")
df.loc[df['relh'] > 120, 'relh'] = np.nan

# Rule B: RH 100% to 120% -> Cap at 100%
#high_rh_count = len(df[(df['relh'] > 100) & (df['relh'] <= 120)])
#print(f"  - Found {high_rh_count} records with RH between 100-120%. Capping at 100%.")
#df.loc[(df['relh'] > 100) & (df['relh'] <= 120), 'relh'] = 100.0

# --- 3. AGGREGATE TO DAILY ---
print("Aggregating to Daily Averages...")
# We use min_count=1 so if all hours are NaN, the result is NaN (not 0)
df_daily = df.groupby(['station', 'date'])[['relh', 'wind_ms']].mean().reset_index()

# ---> NEW ADDITION: Save the daily data to Drive <---
daily_master_path = os.path.join(output_folder, 'ASOS_Daily_Data_Master.csv')
df_daily.to_csv(daily_master_path, index=False)
print(f"  - SUCCESS: Daily master data saved to: {daily_master_path}")
# ----------------------------------------------------

# --- 4. IDENTIFY MISSING DATA (The Issues List) ---
print("Generating Issues List...")

# Filter for days where either RH or Wind is NaN
issues = df_daily[df_daily['relh'].isna() | df_daily['wind_ms'].isna()].copy()

# Add a description column
conditions = [
    (issues['relh'].isna()) & (issues['wind_ms'].isna()),
    (issues['relh'].isna()),
    (issues['wind_ms'].isna())
]
choices = ['Both Missing', 'Humidity Missing', 'Wind Missing']
issues['issue_type'] = np.select(conditions, choices, default='Unknown')

# Save to CSV
issues_path = os.path.join(output_folder, 'ASOS_Missing_Data_Report.csv')
issues.to_csv(issues_path, index=False)
print(f"  - Issues list saved to: {issues_path}")
print(f"  - Total missing daily records: {len(issues)}")

# --- 5. VISUALIZATION (Dynamic Timeline) ---
sns.set_theme(style="whitegrid")

def plot_final_quality(df, var_name, unit, title, color_code):
    # A. Box Plot
    plt.figure(figsize=(10, 6))
    sns.boxplot(x='station', y=var_name, data=df, color=color_code)
    plt.title(f'Distribution: {title}')
    plt.show()

    # B. Time Series (Dynamic X-Axis)
    df_monthly = df.set_index('date').groupby('station')[var_name].resample('M').mean().reset_index()

    plt.figure(figsize=(14, 6))
    sns.lineplot(data=df_monthly, x='date', y=var_name, hue='station', palette='viridis', alpha=0.8)

    # DYNAMIC AXIS: This ensures it shows from your start date to your latest record
    plt.xlim(df['date'].min(), df['date'].max())

    plt.title(f'Temporal Trend: {title} (Monthly Avg)')
    plt.ylabel(f'{title} ({unit})')
    plt.savefig(os.path.join(output_folder, f'Final_{title.replace(" ", "_")}_timeseries.png'), dpi=300)
    plt.show()

# Run the plots
plot_final_quality(df_daily, 'relh', '%', 'Relative Humidity', 'lightblue')
plot_final_quality(df_daily, 'wind_ms', 'm/s', 'Wind Speed', 'lightgreen')

In [ ]:
df_daily.tail(100)

,station,date,relh,wind_ms
31013,OVL,2025-10-21,67.845833,8.703931
31014,OVL,2025-10-22,72.368056,5.947805
31015,OVL,2025-10-23,71.168889,2.677583
31016,OVL,2025-10-24,63.663194,2.834668
31017,OVL,2025-10-25,55.945694,5.212362
...,...,...,...,...
31108,OVL,2026-01-26,54.255000,6.940296
31109,OVL,2026-01-27,63.723194,7.318728
31110,OVL,2026-01-28,63.195833,4.626864
31111,OVL,2026-01-29,68.969583,2.149207


#IEM 2

This will find missing data and fill with average from remaining stations or -99.0.

In [ ]:
import pandas as pd
import numpy as np
import os

# --- CONFIGURATION ---
input_file = '/content/drive/My Drive/SWAT_High-Island/SWAT_weather_files/ASOS_Daily_Data_Master.csv'
output_folder = '/content/drive/My Drive/SWAT_High-Island/SWAT_weather_files'

# Subfolders for SWAT files
for folder in ['wnd', 'rhd', 'review']:
    os.makedirs(os.path.join(output_folder, folder), exist_ok=True)

# --- 1. LOAD DATA & SET TIMELINE ---
print("Loading daily master data...")
df_daily = pd.read_csv(input_file, parse_dates=['date'])

# Create a continuous date range to expose totally missing days
start_date = df_daily['date'].min()
end_date = df_daily['date'].max()
full_dates = pd.date_range(start=start_date, end=end_date, freq='D')
start_date_str = start_date.strftime('%Y%m%d')

print(f"Timeline: {start_date.date()} to {end_date.date()}")

# --- 2. PIVOT & GAP-FILL RELATIVE HUMIDITY ---
print("\nProcessing Relative Humidity...")
# Pivot so stations are columns, dates are rows
rh_pivot = df_daily.pivot(index='date', columns='station', values='relh').reindex(full_dates)

# Step A: Calculate daily averages across available stations (ignores NaNs)
rh_daily_mean = rh_pivot.mean(axis=1)

# Step B: Fill missing station data with that day's average
rh_filled = rh_pivot.apply(lambda col: col.fillna(rh_daily_mean))

# Step C: If all stations were missing, the mean is NaN. Fill those with -99.0
rh_final = rh_filled.fillna(-99.0)

# Save a review file so you can see the filled data side-by-side
rh_review_path = os.path.join(output_folder, 'review', 'RH_Filled_Review.csv')
rh_final.to_csv(rh_review_path, float_format='%.3f')
print(f"  - Saved RH review file to: {rh_review_path}")

# --- 3. PIVOT & GAP-FILL WIND SPEED ---
print("Processing Wind Speed...")
wind_pivot = df_daily.pivot(index='date', columns='station', values='wind_ms').reindex(full_dates)

# Same 3-step gap-filling process
wind_daily_mean = wind_pivot.mean(axis=1)
wind_filled = wind_pivot.apply(lambda col: col.fillna(wind_daily_mean))
wind_final = wind_filled.fillna(-99.0)

# Save a review file
wind_review_path = os.path.join(output_folder, 'review', 'Wind_Filled_Review.csv')
wind_final.to_csv(wind_review_path, float_format='%.3f')
print(f"  - Saved Wind review file to: {wind_review_path}")


# --- 4. GENERATE INDIVIDUAL SWAT+ FILES (3 Stations x 2 Variables = 6 Files) ---
print("\nGenerating final 6 SWAT+ text files...")
stations = rh_final.columns

for station in stations:
    safe_name = str(station).replace(" ", "_").replace("-", "")

    # A. RELATIVE HUMIDITY (.rhd)
    # Convert to fraction (0.0 to 1.0) and handle the -99.0 flag
    rh_values = rh_final[station].copy()
    # Only divide by 100 if it's a valid measurement, keep -99.0 as -99.0
    rh_values = np.where(rh_values != -99.0, rh_values / 100.0, -99.0)

    rh_filepath = os.path.join(output_folder, 'rhd', f'asos_{safe_name}.rhd')
    with open(rh_filepath, 'w') as f:
        f.write(f"{start_date_str}\n")
        np.savetxt(f, rh_values, fmt='%.3f')

    # B. WIND SPEED (.wnd)
    wind_values = wind_final[station].values
    wind_filepath = os.path.join(output_folder, 'wnd', f'asos_{safe_name}.wnd')
    with open(wind_filepath, 'w') as f:
        f.write(f"{start_date_str}\n")
        np.savetxt(f, wind_values, fmt='%.3f')

print(f"\nSUCCESS! All 6 files created in {output_folder}/wnd and {output_folder}/rhd.")

Loading daily master data...
Timeline: 1992-08-04 to 2026-01-30

Processing Relative Humidity...
  - Saved RH review file to: /content/drive/My Drive/SWAT_High-Island/SWAT_weather_files/review/RH_Filled_Review.csv
Processing Wind Speed...
  - Saved Wind review file to: /content/drive/My Drive/SWAT_High-Island/SWAT_weather_files/review/Wind_Filled_Review.csv

Generating final 6 SWAT+ text files...

SUCCESS! All 6 files created in /content/drive/My Drive/SWAT_High-Island/SWAT_weather_files/wnd and /content/drive/My Drive/SWAT_High-Island/SWAT_weather_files/rhd.


#ERA-5 dataset
```
Solar radiation (J/m2 -> MJ/m2)
```

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# --- CONFIGURATION ---
# Update this filename if you named it something else in GEE
input_file = '/content/drive/My Drive/SWAT_High-Island/SWAT_Solar_Radiation_ERA5.csv'
output_folder = '/content/drive/My Drive/SWAT_High-Island/SWAT_weather_files'

# --- 1. LOAD AND PREPARE ---
print(f"Loading {input_file}...")
try:
    df = pd.read_csv(input_file)
except FileNotFoundError:
    print("ERROR: File not found! Please run the GEE script and upload the CSV to Drive.")
    raise

# Inspect columns to ensure we pick the right ones
print(f"Columns found: {df.columns.tolist()}")

# The GEE script likely output 'date' (YYYYMMdd) and 'solar_rad_MJ' (or 'solar_mj_m2')
# Let's find the column that looks like solar data
solar_col = [c for c in df.columns if 'solar' in c.lower() or 'rad' in c.lower()][0]
date_col = [c for c in df.columns if 'date' in c.lower()][0]

print(f"Using Date Column: '{date_col}' and Solar Column: '{solar_col}'")

# Convert Date Format (YYYYMMdd -> Datetime)
# GEE usually exports as a string or number like 19920101
df['date'] = pd.to_datetime(df[date_col].astype(str), format='%Y%m%d')

# Sort by Date
df = df.sort_values(by='date')

# --- 2. CHECK FOR MISSING DAYS ---
print("\n--- CHECKING FOR GAPS ---")
start_date = df['date'].min()
end_date = df['date'].max()
full_range = pd.date_range(start=start_date, end=end_date, freq='D')

# Reindex to catch missing calendar days
df_indexed = df.set_index('date').reindex(full_range)

# Count NaNs (Missing values)
missing_count = df_indexed[solar_col].isna().sum()
total_days = len(df_indexed)

print(f"Timeline: {start_date.date()} to {end_date.date()}")
print(f"Total Expected Days: {total_days}")
print(f"Missing Days: {missing_count}")

if missing_count > 0:
    print("WARNING: You have missing days! These will be filled with -99.0 for SWAT.")
    # Optional: Fill small gaps with interpolation?
    # df_indexed[solar_col] = df_indexed[solar_col].interpolate(method='linear')
    # print("  > Gaps interpolated linearly.")
else:
    print("Perfect! Continuous record.")

# --- 3. PLOTTING ---
sns.set_theme(style="whitegrid")

# A. Histogram (Distribution)
plt.figure(figsize=(10, 6))
sns.histplot(df_indexed[solar_col], bins=50, kde=True, color='orange')
plt.title('Distribution of Solar Radiation (Daily)')
plt.xlabel('Solar Radiation (MJ/m²)')
plt.ylabel('Frequency')
plt.savefig(os.path.join(output_folder, 'Solar_Radiation_Histogram.png'), dpi=300)
plt.show()

# B. Time Series (Line Graph)
plt.figure(figsize=(14, 6))
# Plotting the raw data (or interpolated if you chose to fill)
plt.plot(df_indexed.index, df_indexed[solar_col], color='orange', linewidth=0.5, label='Daily Solar Rad')

# Add a smoothed line (monthly rolling average) to see the "seasons" better
#rolling_mean = df_indexed[solar_col].rolling(window=30, center=True).mean()
#plt.plot(df_indexed.index, rolling_mean, color='red', linewidth=2, label='30-Day Avg')

plt.title('Solar Radiation Time Series (ERA5)')
plt.ylabel('Solar Radiation (MJ/m²)')
plt.legend()
plt.xlim(start_date, end_date)
plt.savefig(os.path.join(output_folder, 'Solar_Radiation_TimeSeries.png'), dpi=300)
plt.show()

# --- 4. GENERATE SWAT+ FILE ---
print("\n--- WRITING SWAT+ INPUT FILE ---")

# Fill NaNs with -99.0 (SWAT code for missing) just in case
df_final = df_indexed.fillna(-99.0)

# Define filename (e.g., 'solar_era5.slr')
swat_filename = os.path.join(output_folder, 'solar_era5.slr')

# Get start date string for Header (e.g., 19920101)
header_date = start_date.strftime('%Y%m%d')

with open(swat_filename, 'w') as f:
    # Line 1: Start Date
    f.write(f"{header_date}\n")

    # Line 2+: Values (One per line)
    # Format to 3 decimal places
    for val in df_final[solar_col]:
        f.write(f"{val:.3f}\n")

print(f"SUCCESS! SWAT+ file created at: {swat_filename}")
print(f"Header: {header_date}")
print(f"First 5 values:\n{df_final[solar_col].head().to_string(index=False)}")

#[old - do not use] IEM 2
This will create 3 files:

**File 1 (Master Continuous Record):** It will force every station to have a row for every single day of the timeline. If data is missing, it will be empty (NaN), but the row will exist.
Use it for: Checking specific dates manually. This is the base file we will eventually use to generate the final SWAT+ inputs.

**File 2 (The "Fixable" Gaps):** It scans each day. If Station A is missing, but Station B or C has data, it calculates the average from the neighbors and offers that as the solution.
Use it for: Validating that filling makes sense. If you see the "potential value" is wildly different from what you expect, check that day.

**File 3 (The "Black Holes"):** It finds days where all 3 stations are missing data. These are the days where you cannot use an average; the SWAT+ Weather Generator will have to invent data for these.

In [ ]:
import pandas as pd
import numpy as np
import os

# --- CONFIGURATION ---
input_file = '/content/drive/My Drive/SWAT_High-Island/ASOS_data.txt'
output_folder = '/content/drive/My Drive/SWAT_High-Island/SWAT_weather_files'

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# --- 1. LOAD AND CLEAN (Same as before) ---
print("Loading and cleaning raw data...")
df = pd.read_csv(input_file, parse_dates=['valid'], na_values=['M', 'm', 'Missing'])
df.replace({'T': 0.001}, inplace=True)
df['relh'] = pd.to_numeric(df['relh'], errors='coerce')
df['sped'] = pd.to_numeric(df['sped'], errors='coerce')
df['wind_ms'] = df['sped'] * 0.44704
df['date'] = pd.to_datetime(df['valid'].dt.date)

# Apply Cleaning Rule: RH > 120% is NaN (Sensor Error)
df.loc[df['relh'] > 120, 'relh'] = np.nan
# Note: We are keeping 100-120% as is, per your request.

# Aggregate to Daily
df_daily = df.groupby(['station', 'date'])[['relh', 'wind_ms']].mean().reset_index()

# --- 2. SETUP TIMELINES ---
# Get list of stations
stations = df_daily['station'].unique()
print(f"Stations found: {stations}")

# Determine the "Active Start Date" for each station
# (The first day they actually reported data)
station_starts = df_daily.groupby('station')['date'].min()
print("\nStation Start Dates:")
print(station_starts)

# Create Global Timeline
global_start = df_daily['date'].min()
global_end = df_daily['date'].max()
full_dates = pd.date_range(global_start, global_end, freq='D')

# --- 3. GENERATE FILE 1: MASTER CONTINUOUS RECORD ---
print("\nGenerating File 1 (Continuous Record)...")
master_records = []

for station in stations:
    # Get data for this station
    st_data = df_daily[df_daily['station'] == station].set_index('date')

    # Reindex to full timeline (fills missing days with NaN)
    st_reindexed = st_data.reindex(full_dates)
    st_reindexed['station'] = station
    st_reindexed.index.name = 'date'

    # Reset index to make date a column
    master_records.append(st_reindexed.reset_index())

df_master = pd.concat(master_records, ignore_index=True)
# Sort for cleanliness
df_master = df_master.sort_values(by=['date', 'station'])

save_path_1 = os.path.join(output_folder, '1_Master_Continuous_Record.csv')
df_master.to_csv(save_path_1, index=False, float_format='%.2f')
print(f"Saved File 1: {save_path_1}")


# --- PREPARE FOR FILES 2 & 3 (PIVOT TABLE STRATEGY) ---
# We pivot the data so stations are columns side-by-side.
# This makes comparing them easy.
pivot_rh = df_daily.pivot(index='date', columns='station', values='relh').reindex(full_dates)
pivot_wind = df_daily.pivot(index='date', columns='station', values='wind_ms').reindex(full_dates)

# Lists to store our findings
fillable_gaps = []
total_failures = []

# --- 4. ANALYZE GAPS DAY-BY-DAY ---
print("\nAnalyzing gaps for Files 2 & 3...")

for current_date in full_dates:
    # Identify which stations SHOULD be active today
    active_stations = [st for st in stations if current_date >= station_starts[st]]

    # If no stations are active yet (shouldn't happen with full_dates), skip
    if not active_stations:
        continue

    # --- CHECK RELATIVE HUMIDITY ---
    # Get values for active stations only
    rh_values = pivot_rh.loc[current_date, active_stations]

    # Count how many are NaN
    missing_count = rh_values.isna().sum()
    available_count = rh_values.notna().sum()

    if missing_count > 0:
        if available_count > 0:
            # PARTIAL FAILURE (File 2): We can fill this!
            # Calculate the average of available stations
            avg_val = rh_values.mean()
            fillable_gaps.append({
                'date': current_date,
                'variable': 'Relative_Humidity',
                'stations_missing': rh_values[rh_values.isna()].index.tolist(),
                'stations_available': rh_values[rh_values.notna()].index.tolist(),
                'potential_fill_value': avg_val
            })
        else:
            # TOTAL FAILURE (File 3): No data exists at all
            total_failures.append({
                'date': current_date,
                'variable': 'Relative_Humidity',
                'active_stations_checked': active_stations
            })

    # --- CHECK WIND SPEED ---
    # Same logic
    wind_values = pivot_wind.loc[current_date, active_stations]

    missing_count_w = wind_values.isna().sum()
    available_count_w = wind_values.notna().sum()

    if missing_count_w > 0:
        if available_count_w > 0:
            # PARTIAL FAILURE (File 2)
            avg_val_w = wind_values.mean()
            fillable_gaps.append({
                'date': current_date,
                'variable': 'Wind_Speed',
                'stations_missing': wind_values[wind_values.isna()].index.tolist(),
                'stations_available': wind_values[wind_values.notna()].index.tolist(),
                'potential_fill_value': avg_val_w
            })
        else:
            # TOTAL FAILURE (File 3)
            total_failures.append({
                'date': current_date,
                'variable': 'Wind_Speed',
                'active_stations_checked': active_stations
            })

# --- 5. SAVE FILE 2: FILLABLE GAPS ---
df_fillable = pd.DataFrame(fillable_gaps)
if not df_fillable.empty:
    save_path_2 = os.path.join(output_folder, '2_Fillable_Gaps_Report.csv')
    df_fillable.to_csv(save_path_2, index=False, float_format='%.2f')
    print(f"Saved File 2: {save_path_2} (Found {len(df_fillable)} fillable instances)")
else:
    print("File 2 Empty: No fillable gaps found.")

# --- 6. SAVE FILE 3: TOTAL FAILURES ---
df_failures = pd.DataFrame(total_failures)
if not df_failures.empty:
    save_path_3 = os.path.join(output_folder, '3_Total_Failure_Report.csv')
    df_failures.to_csv(save_path_3, index=False)
    print(f"Saved File 3: {save_path_3} (Found {len(df_failures)} days with NO data)")
else:
    print("File 3 Empty: Amazing! You have complete coverage.")

print("\nProcessing Complete.")

Loading and cleaning raw data...
Stations found: ['GYL' 'HCD' 'OVL']

Station Start Dates:
station
GYL   1998-10-31
HCD   1992-08-04
OVL   2001-04-11
Name: date, dtype: datetime64[ns]

Generating File 1 (Continuous Record)...
Saved File 1: /content/drive/My Drive/SWAT_High-Island/SWAT_weather_files/1_Master_Continuous_Record.csv

Analyzing gaps for Files 2 & 3...
Saved File 2: /content/drive/My Drive/SWAT_High-Island/SWAT_weather_files/2_Fillable_Gaps_Report.csv (Found 380 fillable instances)
Saved File 3: /content/drive/My Drive/SWAT_High-Island/SWAT_weather_files/3_Total_Failure_Report.csv (Found 17 days with NO data)

Processing Complete.


#[old - do not use] Error checking

In [ ]:
#Daily_Diagnostic files for each station.

import pandas as pd
import numpy as np
import os

# --- CONFIGURATION ---
input_file = '/content/drive/My Drive/SWAT_High-Island/ASOS_data.txt'
output_folder = '/content/drive/My Drive/SWAT_High-Island/SWAT_weather_files'

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# --- 1. LOAD AND CLEAN ---
print("Reading raw file...")
# We use 'na_values' to tell pandas exactly what looks like a missing value right at the start
df = pd.read_csv(input_file, parse_dates=['valid'], na_values=['M', 'm', 'Missing'])

# Replace Trace 'T' with a tiny number
df.replace({'T': 0.001}, inplace=True)

# Force numeric conversion (This fixes the "Text vs Number" issue)
# If a value is unreadable, it becomes NaN
df['relh'] = pd.to_numeric(df['relh'], errors='coerce')
df['sped'] = pd.to_numeric(df['sped'], errors='coerce')

# Convert Wind to m/s
df['wind_ms'] = df['sped'] * 0.44704

# Extract just the Date
df['date'] = df['valid'].dt.date
df['date'] = pd.to_datetime(df['date'])

# Get list of unique stations
stations = df['station'].unique()
print(f"Found {len(stations)} stations: {stations}")

# Define global start/end dates to ensure all files span the same period
global_start = df['date'].min()
global_end = df['date'].max()
full_date_range = pd.date_range(start=global_start, end=global_end, freq='D')

# --- 2. PROCESS EACH STATION ---
for station in stations:
    print(f"\nProcessing Station: {station}...")

    # Filter data for this station
    st_data = df[df['station'] == station].copy()

    # AGGREGATE TO DAILY
    # min_count=1 ensures that if all hours are NaN, the day is NaN (not 0)
    daily = st_data.groupby('date')[['relh', 'wind_ms']].agg(lambda x: x.mean(skipna=True))

    # REINDEX to fill in missing calendar days
    daily = daily.reindex(full_date_range)
    daily.index.name = 'Date'

    # Reset index to make Date a column
    daily = daily.reset_index()

    # --- 3. CREATE DIAGNOSTIC FLAGS ---
    # We create a list to store the flag for each row
    flags = []

    for index, row in daily.iterrows():
        issues = []

        # Check Relative Humidity
        if pd.isna(row['relh']):
            issues.append("MISSING_RH")
        elif row['relh'] == 0:
            issues.append("RH_ZERO") # Suspicious zero
        elif row['relh'] > 100:
            issues.append("RH_HIGH") # > 100%

        # Check Wind
        if pd.isna(row['wind_ms']):
            issues.append("MISSING_WIND")

        # Combine flags
        if not issues:
            flags.append("GOOD")
        else:
            flags.append("; ".join(issues))

    daily['Quality_Flag'] = flags

    # --- 4. SAVE TO CSV ---
    filename = f"{station}_Daily_Diagnostic.csv"
    save_path = os.path.join(output_folder, filename)

    # Save with 2 decimal places for cleanliness
    daily.to_csv(save_path, index=False, float_format='%.2f')
    print(f"Saved: {filename}")

    # --- Quick Report ---
    print(f"  > Total Days: {len(daily)}")
    print(f"  > Good Days: {len(daily[daily['Quality_Flag'] == 'GOOD'])}")
    print(f"  > Missing Data Days: {daily['Quality_Flag'].str.contains('MISSING').sum()}")
    print(f"  > Zero Humidity Days: {daily['Quality_Flag'].str.contains('RH_ZERO').sum()}")

print("\nProcessing complete! Check your Drive folder.")

Reading raw file...
Found 3 stations: ['HCD' 'GYL' 'OVL']

Processing Station: HCD...
Saved: HCD_Daily_Diagnostic.csv
  > Total Days: 12233
  > Good Days: 12158
  > Missing Data Days: 70
  > Zero Humidity Days: 0

Processing Station: GYL...
Saved: GYL_Daily_Diagnostic.csv
  > Total Days: 12233
  > Good Days: 9836
  > Missing Data Days: 2392
  > Zero Humidity Days: 0

Processing Station: OVL...
Saved: OVL_Daily_Diagnostic.csv
  > Total Days: 12233
  > Good Days: 8991
  > Missing Data Days: 3242
  > Zero Humidity Days: 0

Processing complete! Check your Drive folder.


In [ ]:
#THIS CODE SHOWED THE PROBLEM OF 3 QUADRILLION RELATIVE HUMIDITY ON 3 DAYS. Plotting each of the Daily_Diagnostic e analyzing the data quality.
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# --- CONFIGURATION ---
output_folder = '/content/drive/My Drive/SWAT_High-Island/SWAT_weather_files'

# The specific files you mentioned
files = [
    'OVL_Daily_Diagnostic.csv',
    'HCD_Daily_Diagnostic.csv',
    'GYL_Daily_Diagnostic.csv'
]

sns.set_theme(style="whitegrid")

for filename in files:
    file_path = os.path.join(output_folder, filename)
    station_name = filename.split('_')[0]

    print(f"\n{'='*40}")
    print(f"ANALYZING STATION: {station_name}")
    print(f"{'='*40}")

    if not os.path.exists(file_path):
        print(f"ERROR: Could not find {filename}. Skipping.")
        continue

    # Load Data
    df = pd.read_csv(file_path)
    df['Date'] = pd.to_datetime(df['Date'])

    # --- 1. THE TRUTH TABLE ---
    total_days = len(df)
    missing_vals = df['relh'].isna().sum()
    zeros = len(df[df['relh'] == 0])
    valid_data = total_days - missing_vals - zeros

    print(f"Total Days:      {total_days}")
    print(f"Missing (NaN):   {missing_vals}")
    print(f"Zeros (0.00):    {zeros}  <-- THESE ARE THE CULPRITS!")
    print(f"Real Data (>0):  {valid_data}")

    if valid_data > 0:
        print(f"Sample of valid data:\n{df[df['relh'] > 0]['relh'].head().tolist()}")
        print(f"Data Range: {df[df['relh'] > 0]['relh'].min()} to {df[df['relh'] > 0]['relh'].max()}")
    else:
        print("WARNING: This station has NO valid humidity data greater than 0.")

    # --- 2. PLOTTING ---
    # We create TWO plots for each station
    fig, axes = plt.subplots(2, 1, figsize=(12, 10))
    fig.suptitle(f"Deep Dive: {station_name} Relative Humidity", fontsize=16)

    # Plot A: The Raw Data (shows the zeros)
    sns.lineplot(data=df, x='Date', y='relh', ax=axes[0], color='red', alpha=0.5)
    axes[0].set_title(f"Raw Data (Includes Zeros)")
    axes[0].set_ylabel("Humidity (%)")

    # Plot B: The Filtered Data (IGNORES zeros)
    # This is the "Truth" plot - if the wave exists, it will appear here
    clean_df = df[df['relh'] > 0.1].copy() # Filter out zeros

    if not clean_df.empty:
        sns.scatterplot(data=clean_df, x='Date', y='relh', ax=axes[1], color='blue', s=10)
        axes[1].set_title(f"Cleaned Data (Zeros Removed)")
        axes[1].set_ylabel("Humidity (%)")
        axes[1].set_ylim(0, 110) # Force scale to normal range
    else:
        axes[1].text(0.5, 0.5, "NO VALID DATA TO PLOT", ha='center', fontsize=14)

    plt.tight_layout()
    plt.show()